# Proyecto: Clasificación de Textos en Lenguaje Natural
## Autor: Raúl González Acosta ([alu0101543529](mailto:alu0101543529@ull.edu.es))

### Clasificación y Generación de CSV

**Objetivo:** Utilizar los tres modelos entrenados para clasificar un corpus de entrada y generar un archivo CSV con las predicciones.

**Descripción:** Esta fase final utiliza los modelos que ya previamente han sido entrenados para:
1. Evaluar el rendimiento de los modelos en el conjunto de prueba
2. Cargar y preprocesar el corpus de entrada
3. Generar predicciones de los tres modelos para cada texto del corpus
4. Exportar los resultados en formato CSV

El archivo CSV contiene:
- **Índice:** Número de fila (1, 2, 3, ...)
- **pad:** Predicción del modelo Dense
- **recurrente:** Predicción del modelo RNN/LSTM
- **transformer:** Predicción del modelo Transformer

Cada predicción es 'P' para Poesía o 'R' para Rap. Los textos se mantienen en el mismo orden que en el archivo de entrada.

In [21]:
import csv
import json
import math
import random
import re
import unicodedata
from pathlib import Path

import nltk
import numpy as np
import torch
import torch.nn as nn
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

nltk.download('stopwords', quiet=True)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

PROJECT_DIR = Path('./')
CORPUS_DIR = PROJECT_DIR / 'corpus'
DATA_DIR = PROJECT_DIR / 'datos_preprocesados'
MODELS_DIR = PROJECT_DIR / 'modelos_entrenados'
OUTPUT_CSV = PROJECT_DIR / 'alu0101543529.csv'
INPUT_CORPUS_PATH = CORPUS_DIR / 'poesia.txt'

with open(DATA_DIR / 'vocabulary.json', 'r', encoding='utf-8') as f:
    vocab_data = json.load(f)
    word_to_idx = vocab_data['word_to_idx']

with open(MODELS_DIR / 'models_info.json', 'r', encoding='utf-8') as f:
    models_info = json.load(f)

print(f'Vocabulario: {len(word_to_idx)} palabras')

def load_corpus(filepath: Path) -> list[str]:
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()
    texts = []
    for match in re.findall(r'<CS>(.*?)</CS>', content, flags=re.S):
        text = match.replace('<EOL>', ' ')
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            texts.append(text)
    return texts

if not INPUT_CORPUS_PATH.exists():
    raise FileNotFoundError(f'No existe el corpus de entrada: {INPUT_CORPUS_PATH}')

raw_texts = load_corpus(INPUT_CORPUS_PATH)
print(f'Corpus de entrada cargado: {len(raw_texts)} textos')

Dispositivo: cuda
Vocabulario: 32917 palabras
Corpus de entrada cargado: 4000 textos


## 2. Preprocesamiento Idéntico al de Entrenamiento

Se aplica el mismo pipeline de preprocesamiento utilizado anteriormente:
- El corpus de entrada (textos a clasificar)
- Los datos de entrenamiento y prueba (para consistencia)

El preprocesamiento idéntico es crítico para asegurar que los modelos reciban datos en el mismo formato que usaron durante el entrenamiento. Cualquier diferencia en el preprocesamiento podría degradar significativamente el rendimiento.

In [22]:
class TextPreprocessor:
    def __init__(self):
        self.stop_words = set(stopwords.words('spanish'))
        self.stemmer = SnowballStemmer('spanish')

    def normalize_unicode(self, text: str) -> str:
        normalized = unicodedata.normalize('NFKD', text)
        return normalized.encode('ascii', 'ignore').decode('ascii')

    def to_lowercase(self, text: str) -> str:
        return text.lower()

    def remove_punctuation(self, text: str) -> str:
        return re.sub(r'[^\w\s]', ' ', text)

    def remove_numbers(self, text: str) -> str:
        return re.sub(r'\d+', ' ', text)

    def tokenize(self, text: str) -> list[str]:
        return re.findall(r'\b\w+\b', text, flags=re.UNICODE)

    def remove_stopwords(self, tokens: list[str]) -> list[str]:
        return [token for token in tokens if token not in self.stop_words and len(token) > 1]

    def stem(self, tokens: list[str]) -> list[str]:
        return [self.stemmer.stem(token) for token in tokens]

    def preprocess(self, text: str) -> list[str]:
        text = self.normalize_unicode(text)
        text = self.to_lowercase(text)
        text = self.remove_punctuation(text)
        text = self.remove_numbers(text)
        tokens = self.tokenize(text)
        tokens = self.remove_stopwords(tokens)
        tokens = self.stem(tokens)
        return tokens


preprocessor = TextPreprocessor()
input_texts = [preprocessor.preprocess(text) for text in raw_texts]

print('Ejemplo preprocesado:')
print(input_texts[0][:30] if input_texts else [])

class TextClassificationDataset(Dataset):
    def __init__(self, texts, word_to_idx=None):
        self.texts = texts
        self.word_to_idx = word_to_idx or {}

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        indices = torch.tensor([self.word_to_idx.get(word, 0) for word in text], dtype=torch.long)
        return indices if len(indices) else torch.tensor([0], dtype=torch.long)


def collate_fn_pad(batch, max_length=None):
    sequences = list(batch)
    if max_length is not None:
        sequences = [seq[:max_length] for seq in sequences]
    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_sequences, lengths

Ejemplo preprocesado:
['parqu', 'confus', 'langu', 'bris', 'ciel', 'sahum', 'cipr', 'hus', 'devan', 'ovill', 'brum', 'tel', 'lun', 'tiend', 'plat', 'urdimbr', 'abandon', 'rad', 'lugubr', 'corsari', 'despu', 'suen', 'timbr', 'vecindari', 'horizont', 'malv', 'mar', 'argentin', 'curv', 'frent']


## 3. Modelos y Carga de Checkpoints

También, se definen las arquitecturas de los modelos y se cargan los pesos guardados. La concordancia entre arquitecturas y pesos es esencial para poder usar los modelos entrenados.

In [23]:
def build_loader(texts, batch_size=32, shuffle=False, max_length=None):
    dataset = TextClassificationDataset(texts, word_to_idx)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=lambda batch: collate_fn_pad(batch, max_length=max_length))


def positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    position = torch.arange(max_len).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe
  
class DenseTextClassifier(nn.Module):
    def __init__(self, vocab_size, context_size, embed_size, hidden_dim):
        super().__init__()
        self.context_size = context_size
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.fc1 = nn.Linear(embed_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        summed = torch.sum(emb, dim=1)
        hidden = torch.relu(self.fc1(summed))
        return self.fc2(hidden).squeeze(-1)


class RNNTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        emb = self.embedding(x)
        _, hidden = self.rnn(emb)
        return self.fc(hidden[-1]).squeeze(-1)


class FeedForward(nn.Module):
    def __init__(self, emb_dim, hidden_dim=None):
        super().__init__()
        hidden_dim = hidden_dim or emb_dim * 4
        self.fc1 = nn.Linear(emb_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, emb_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


class DecoderBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, hidden_dim=None, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=emb_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.ffn = FeedForward(emb_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(emb_dim)
        self.norm2 = nn.LayerNorm(emb_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, causal_mask, padding_mask=None):
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed, attn_mask=causal_mask, key_padding_mask=padding_mask, need_weights=False)
        x = x + self.drop(attn_out)
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class TransformerTextClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_size, num_heads, hidden_dim, num_layers, num_classes, pad_idx, dropout=0.1):
        super().__init__()
        self.context_size = context_size
        self.emb_dim = emb_dim
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.register_buffer('pe_buffer', positional_encoding(context_size, emb_dim))
        self.pos_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList([DecoderBlock(emb_dim, num_heads, hidden_dim, dropout) for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(emb_dim)
        self.classifier_head = nn.Linear(emb_dim, num_classes)

    def _make_causal_mask(self, seq_len, device):
        return torch.triu(torch.ones((seq_len, seq_len), dtype=torch.bool, device=device), diagonal=1)

    def forward(self, src, padding_mask=None):
        batch_size, seq_len = src.size()
        x = self.embedding(src) * math.sqrt(self.emb_dim)
        x = self.pos_dropout(x + self.pe_buffer[:seq_len, :].unsqueeze(0))
        causal_mask = self._make_causal_mask(seq_len, src.device)
        for layer in self.layers:
            x = layer(x, causal_mask, padding_mask)
        x = self.final_norm(x)
        if padding_mask is not None:
            seq_lengths = (~padding_mask).sum(dim=1)
            last_real_idx = torch.clamp(seq_lengths - 1, min=0)
        else:
            last_real_idx = torch.full((batch_size,), seq_len - 1, dtype=torch.long, device=src.device)
        batch_idx = torch.arange(batch_size, device=src.device)
        last_token_repr = x[batch_idx, last_real_idx, :]
        return self.classifier_head(last_token_repr)


def load_model_weights(model, checkpoint_path: Path):
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'No se encuentra el checkpoint: {checkpoint_path}')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    state_dict = checkpoint['model_state_dict'] if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict(state_dict)
    return model


def predict_model(model, texts, batch_size=32, max_length=None):
    loader = build_loader(texts, batch_size=batch_size, shuffle=False, max_length=max_length)
    predictions = []
    model.eval()
    with torch.no_grad():
        for input_ids, lengths in loader:
            input_ids = input_ids.to(device)
            lengths = lengths.to(device)
            if isinstance(model, TransformerTextClassifier):
                seq_len = input_ids.size(1)
                padding_mask = torch.arange(seq_len, device=device).unsqueeze(0) >= lengths.unsqueeze(1)
                logits = model(input_ids, padding_mask=padding_mask)
                batch_preds = torch.argmax(logits, dim=1).cpu().tolist()
            else:
                logits = model(input_ids)
                batch_preds = (logits > 0).long().cpu().tolist()
            predictions.extend(batch_preds)
    return [0 if pred == 0 else 1 for pred in predictions]

## 4. Evaluación en Conjunto de Prueba y Predicción del Corpus Objetivo

Para la clasificación, se realizan dos operaciones principales:

1. **Evaluación:** Se calcula el rendimiento de cada modelo en el conjunto de prueba etiquetado, mostrando las métricas finales (accuracy, precision, recall, F1-score).

2. **Predicción:** Se generan predicciones para cada texto del corpus de entrada usando los tres modelos, preservando el orden original de los textos.

Finalmente, se exportan los resultados en un archivo CSV con una fila por texto y columnas para cada modelo.

In [24]:
LABEL_MAP = {0: 'P', 1: 'R'}
dense_params = models_info['Dense']['params']
rnn_params = models_info['RNN']['params']
transformer_params = models_info['Transformer']['params']

dense_model = load_model_weights(
    DenseTextClassifier(
        vocab_size=dense_params['vocab_size'],
        context_size=dense_params['context_size'],
        embed_size=dense_params['embed_size'],
        hidden_dim=dense_params['hidden_dim'],
    ),
    MODELS_DIR / 'Dense_model.pth',
).to(device).eval()

rnn_model = load_model_weights(
    RNNTextClassifier(
        vocab_size=rnn_params['vocab_size'],
        embed_size=rnn_params['embed_size'],
        hidden_dim=rnn_params['hidden_dim'],
    ),
    MODELS_DIR / 'RNN_model.pth',
).to(device).eval()

transformer_model = load_model_weights(
    TransformerTextClassifier(
        vocab_size=transformer_params['vocab_size'],
        emb_dim=transformer_params['embedding_dim'],
        context_size=transformer_params['max_seq_length'],
        num_heads=transformer_params['num_heads'],
        hidden_dim=transformer_params['feed_forward_dim'],
        num_layers=transformer_params['num_layers'],
        num_classes=2,
        pad_idx=0,
        dropout=transformer_params['dropout'],
    ),
    MODELS_DIR / 'Transformer_model.pth',
).to(device).eval()

pad_predictions = [LABEL_MAP[p] for p in predict_model(dense_model, input_texts, batch_size=32, max_length=dense_params['context_size'])]
rnn_predictions = [LABEL_MAP[p] for p in predict_model(rnn_model, input_texts, batch_size=32, max_length=dense_params['context_size'])]
transformer_predictions = [LABEL_MAP[p] for p in predict_model(transformer_model, input_texts, batch_size=4, max_length=transformer_params['max_seq_length'])]

rows = [[index, pad_pred, rnn_pred, transformer_pred] for index, (pad_pred, rnn_pred, transformer_pred) in enumerate(zip(pad_predictions, rnn_predictions, transformer_predictions), start=1)]

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Index', 'pad', 'recurrente', 'transformer'])
    writer.writerows(rows)

print(f'CSV generado en: {OUTPUT_CSV}')
print(f'Textos clasificados: {len(rows)}')
print('Primeras filas:')
for row in rows[:10]:
    print(row)

CSV generado en: alu0101543529.csv
Textos clasificados: 4000
Primeras filas:
[1, 'P', 'P', 'P']
[2, 'P', 'P', 'P']
[3, 'P', 'P', 'P']
[4, 'P', 'P', 'P']
[5, 'P', 'P', 'P']
[6, 'P', 'P', 'P']
[7, 'P', 'P', 'P']
[8, 'P', 'P', 'P']
[9, 'P', 'P', 'P']
[10, 'P', 'P', 'P']
